## 1. Staging

In [1]:
import sys, logging
from os import environ

from microsoft_agents.activity import load_configuration_from_env
from microsoft_agents.authentication.msal import MsalConnectionManager

from datetime import datetime, timedelta

# base graph client dependencies
from azure.core.credentials import AccessToken
from msgraph import GraphServiceClient

# to synthesize query parameters for list incidents and list alerts
from kiota_abstractions.base_request_configuration import RequestConfiguration
from msgraph.generated.security.incidents.incidents_request_builder import IncidentsRequestBuilder
from msgraph.generated.security.alerts_v2.alerts_v2_request_builder import Alerts_v2RequestBuilder

# to synthesize request body for run hunting query
from msgraph.generated.security.microsoft_graph_security_run_hunting_query.run_hunting_query_post_request_body import RunHuntingQueryPostRequestBody

# comments for incident are actually of AlertComment type
from msgraph.generated.models.security.alert_comment import AlertComment

# use for updating incident
from msgraph.generated.models.security.incident import Incident
from msgraph.generated.models.security.incident_status import IncidentStatus
from msgraph.generated.models.security.alert_classification import AlertClassification
from msgraph.generated.models.security.alert_determination import AlertDetermination

logging.basicConfig(
    level=logging.INFO,
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

## 2. Graph token and client
### 2.1. Acquire Graph token for agent user

In [2]:
agents_sdk_config = load_configuration_from_env(environ)
connection_manager = MsalConnectionManager(**agents_sdk_config)
token_provider = connection_manager.get_connection("SERVICE_CONNECTION")

INFO:microsoft_agents.hosting.core.authorization.connection_manager:
Connections:
[
  {
    "CONNECTION": "SERVICE_CONNECTION",
    "CLIENTID": "4d3...",
    "TENANTID": "323...",
    "AUTHORITY": "https://login.microsoftonline.com/...",
    "SCOPES": "... [36 scope(s)]",
    "FEDERATED_CLIENT_ID": "43e...",
    "ANONYMOUS_ALLOWED": "False"
  }
]

Connections Map:
[
  {
    "CONNECTION": "SERVICE_CONNECTION",
    "SERVICEURL": "*"
  }
]


In [3]:
graph_token = await token_provider.get_agentic_user_token(
    tenant_id=environ.get("CONNECTIONS__SERVICE_CONNECTION__SETTINGS__TENANTID"),
    agent_app_instance_id=environ.get("AGENTIC_INSTANCE_ID"),
    agentic_user_id=environ.get("AGENTIC_USER_ID"),
    scopes=["https://graph.microsoft.com/.default"],
)

INFO:microsoft_agents.authentication.msal.msal_auth:Attempting to get agentic user token from agent_app_instance_id ad86a79f-2051-4caf-b11c-ab59e81ddd75 and agentic_user_id 4a1529b5-58a5-4cb3-82f4-29f3cc3012c7
INFO:microsoft_agents.authentication.msal.msal_auth:Attempting to get agentic instance token from agent_app_instance_id ad86a79f-2051-4caf-b11c-ab59e81ddd75
INFO:microsoft_agents.authentication.msal.msal_auth:Attempting to get agentic application token from agent_app_instance_id ad86a79f-2051-4caf-b11c-ab59e81ddd75
INFO:microsoft_agents.authentication.msal.msal_auth:Acquiring agentic user token for agent_app_instance_id ad86a79f-2051-4caf-b11c-ab59e81ddd75 and agentic_user_id 4a1529b5-58a5-4cb3-82f4-29f3cc3012c7
INFO:microsoft_agents.authentication.msal.msal_auth:Acquired agentic user token response.


In [4]:
print(graph_token)

eyJ0eXAiOiJKV1QiLCJub25jZSI6Ik9OWEt5ckRqMkFUcDBiVTVHVWxBMnR6N3I2YjZwQVRJSWVmRkJSOWNrWHciLCJhbGciOiJSUzI1NiIsIng1dCI6ImZFdHFyaEtUMWJYQUdhZlNkUW9OMXZYVFJwSSIsImtpZCI6ImZFdHFyaEtUMWJYQUdhZlNkUW9OMXZYVFJwSSJ9.eyJhdWQiOiJodHRwczovL2dyYXBoLm1pY3Jvc29mdC5jb20iLCJpc3MiOiJodHRwczovL3N0cy53aW5kb3dzLm5ldC8zMjM2MjZmNS0xYmZlLTQ4Y2QtODkwMi1kZGZkZmQ0NGUxY2UvIiwiaWF0IjoxNzg2NTQyOTU2LCJuYmYiOjE3ODY1NDI5NTYsImV4cCI6MTc4NjU0ODEwMCwiYWNjdCI6MCwiYWNyIjoiMCIsImFjcnMiOlsicDEiLCJ1cm46dXNlcjpyZWdpc3RlcnNlY3VyaXR5aW5mbyIsInBmZHIiXSwiYWlvIjoiQVlRQWUvOGNBQUFBVkxERkxkNGhPZGhHV3F0N3E5ZFgyVkpLVlR4WUt3anBKRGYvS0Zhd2hNZDF4elVpTjB4U2I3UTgxSUFzMDlwNC9FNXRNNDFMckNubW8xU0s2TEljdHMxb0E3aEhHL1NPOUc4b0hwMHhtNmZJYVpoN2ZEeStFd2d5MnZiRVFoTUxMMTR0WW43Kzcrc1hrVStFK0drM0xjd3pRL2psSGdESC82THZBT3RkK2hNPSIsImFwcF9kaXNwbGF5bmFtZSI6IkF1dG9TT0MgSWRlbnRpdHkiLCJhcHBpZCI6ImFkODZhNzlmLTIwNTEtNGNhZi1iMTFjLWFiNTllODFkZGQ3NSIsImFwcGlkYWNyIjoiMiIsImlkdHlwIjoidXNlciIsImlwYWRkciI6IjU3LjE2Mi4xMDQuMjA3IiwibmFtZSI6IkF1dG9TT0MiLCJvaWQiOiI0YTE1MjliNS0

### 2.2. Define custom provider class to use token from agents SDK

In [5]:
class RawAccessTokenProvider:
    """A credential provider that returns a raw access token for the SDK."""
    def __init__(self, token: str):
        self.token = token

    # The SDK calls get_token to retrieve the active Bearer token
    def get_token(self, *scopes, **kwargs) -> AccessToken:
        # Provide token string and an arbitrary future expiration timestamp (in seconds)
        return AccessToken(self.token, expires_on=int((datetime.now().astimezone() + timedelta(hours=1)).timestamp()))

### 2.3. Create credential and initialize the Graph client

In [6]:
credential = RawAccessTokenProvider(graph_token)
graph_client = GraphServiceClient(credentials=credential)

### 2.4.. Simple test to check token

In [7]:
user = await graph_client.me.get()

INFO:httpx:HTTP Request: GET https://graph.microsoft.com/v1.0//me "HTTP/1.1 200 OK"


In [8]:
print(user.display_name)

AutoSOC


## 3. Microsoft Graph security API
### 3.1. List incidents

In [9]:
datetimefilter = (datetime.now().astimezone() - timedelta(hours=24)).isoformat()
query_params = IncidentsRequestBuilder.IncidentsRequestBuilderGetQueryParameters(
    filter = f"createdDateTime ge {datetimefilter}",
    top = 10,
)
request_configuration = RequestConfiguration(
    query_parameters = query_params,
)
incidents = await graph_client.security.incidents.get(request_configuration = request_configuration)

INFO:httpx:HTTP Request: GET https://graph.microsoft.com/v1.0//security/incidents?$filter=createdDateTime%20ge%202026-08-11T14%3A01%3A00.091678%2B00%3A00&$top=10 "HTTP/1.1 200 OK"


In [10]:
print(vars(incidents).keys())

dict_keys(['backing_store', 'additional_data', 'odata_count', 'odata_next_link', 'value'])


In [11]:
print(incidents.value)

[]


### 3.2. Get alerts associated with an incident
#### 3.2.1. Option 1: List incident with filter for incident ID and expand alerts
> Note: [Get incident](https://learn.microsoft.com/en-us/graph/api/security-incident-get) API doesn't work because it can't expand alerts

In [12]:
incident_id = 2002
query_params = IncidentsRequestBuilder.IncidentsRequestBuilderGetQueryParameters(
    filter = f"id eq '{incident_id}'",
    expand = ["alerts"],
)
request_configuration = RequestConfiguration(
    query_parameters = query_params,
)
incident_with_alerts = await graph_client.security.incidents.get(request_configuration = request_configuration)

INFO:httpx:HTTP Request: GET https://graph.microsoft.com/v1.0//security/incidents?$expand=alerts&$filter=id%20eq%20%272002%27 "HTTP/1.1 200 OK"


In [13]:
print(vars(incident_with_alerts).keys())

dict_keys(['backing_store', 'additional_data', 'odata_count', 'odata_next_link', 'value'])


In [14]:
print(incident_with_alerts.value)

[Incident(additional_data={}, id='2002', odata_type=None, alerts=[Alert(additional_data={}, id='sn00b30a13-23e6-4b3d-b94d-aa73c3e5d4c6', odata_type=None, actor_display_name=None, additional_data_property=Dictionary(additional_data={'OriginalAlertProductName': 'Microsoft Sentinel', 'OriginalAlertProviderName': 'Azure Sentinel', 'OriginatingAlertSource': 'Azure Sentinel', 'Trigger Operator': 'GreaterThan', 'Trigger Threshold': '0', 'Correlation Id': UUID('e540db6b-62dd-4ac5-bb52-7005d459eb87'), 'Search Query Results Overall Count': '6', 'Data Sources': '["delta-soc"]', 'Analytic Rule Ids': '["ce733b2e-afca-43cf-a926-23c069b17215"]', 'Event Grouping': 'SingleAlert', 'Analytic Rule Name': 'Anomalous Windows Authentication Failure Activity', 'ProcessedBySentinel': 'True', 'Alert generation status': 'Full alert created', 'SystemAlertId': UUID('d7b9ae14-2802-5b62-6d51-8e8754dd47c1'), 'Query Start Time UTC': datetime.datetime(2026, 8, 10, 22, 44, 56, tzinfo=datetime.timezone.utc), 'Query End T

#### 3.2.2. Option 2: List alerts with filter for incident ID

In [15]:
incident_id = 2002
query_params = Alerts_v2RequestBuilder.Alerts_v2RequestBuilderGetQueryParameters(
    filter = f"incidentId eq '{incident_id}'",
)
request_configuration = RequestConfiguration(
    query_parameters = query_params,
)
alerts_in_incident = await graph_client.security.alerts_v2.get(request_configuration = request_configuration)

INFO:httpx:HTTP Request: GET https://graph.microsoft.com/v1.0//security/alerts_v2?$filter=incidentId%20eq%20%272002%27 "HTTP/1.1 200 OK"


In [16]:
print(vars(alerts_in_incident).keys())

dict_keys(['backing_store', 'additional_data', 'odata_count', 'odata_next_link', 'value'])


In [17]:
print(alerts_in_incident.value)

[Alert(additional_data={}, id='sn00b30a13-23e6-4b3d-b94d-aa73c3e5d4c6', odata_type=None, actor_display_name=None, additional_data_property=Dictionary(additional_data={'OriginalAlertProductName': 'Microsoft Sentinel', 'OriginalAlertProviderName': 'Azure Sentinel', 'OriginatingAlertSource': 'Azure Sentinel', 'Trigger Operator': 'GreaterThan', 'Trigger Threshold': '0', 'Correlation Id': UUID('e540db6b-62dd-4ac5-bb52-7005d459eb87'), 'Search Query Results Overall Count': '6', 'Data Sources': '["delta-soc"]', 'Analytic Rule Ids': '["ce733b2e-afca-43cf-a926-23c069b17215"]', 'Event Grouping': 'SingleAlert', 'Analytic Rule Name': 'Anomalous Windows Authentication Failure Activity', 'ProcessedBySentinel': 'True', 'Alert generation status': 'Full alert created', 'SystemAlertId': UUID('d7b9ae14-2802-5b62-6d51-8e8754dd47c1'), 'Query Start Time UTC': datetime.datetime(2026, 8, 10, 22, 44, 56, tzinfo=datetime.timezone.utc), 'Query End Time UTC': datetime.datetime(2026, 8, 11, 4, 44, 57, tzinfo=dateti

### 3.3. Run threat hunting query
1. Query `SecurityIncident` for the past 7 days

In [18]:
request_body = RunHuntingQueryPostRequestBody(
	query = "SecurityIncident",
    timespan = "P7D",
)
hunt_result = await graph_client.security.microsoft_graph_security_run_hunting_query.post(request_body)

INFO:httpx:HTTP Request: POST https://graph.microsoft.com/v1.0//security/microsoft.graph.security.runHuntingQuery "HTTP/1.1 200 OK"


In [19]:
print(vars(hunt_result).keys())

dict_keys(['backing_store', 'additional_data', 'odata_type', 'results', 'schema'])


In [20]:
print(hunt_result.results)

[HuntingRowResult(additional_data={'TenantId': UUID('f119dae4-df67-44a1-b5c7-caa589bcc8ce'), 'TimeGenerated': datetime.datetime(2026, 8, 6, 0, 3, 25, 463333, tzinfo=datetime.timezone.utc), 'IncidentName': UUID('df9183ab-9f7e-4318-8aee-2e683427f841'), 'Title': 'CC_Sensitive information', 'Description': '', 'Severity': 'Medium', 'Status': 'New', 'Classification': '', 'ClassificationComment': '', 'ClassificationReason': '', 'Owner': '{"objectId":null,"email":null,"assignedTo":null,"userPrincipalName":null}', 'ProviderName': 'Microsoft XDR', 'ProviderIncidentId': '1996', 'FirstActivityTime': datetime.datetime(2026, 8, 5, 22, 55, tzinfo=datetime.timezone.utc), 'LastActivityTime': datetime.datetime(2026, 8, 5, 23, 50, tzinfo=datetime.timezone.utc), 'FirstModifiedTime': None, 'LastModifiedTime': datetime.datetime(2026, 8, 6, 0, 3, 25, 463333, tzinfo=datetime.timezone.utc), 'CreatedTime': datetime.datetime(2026, 8, 6, 0, 3, 25, 463333, tzinfo=datetime.timezone.utc), 'ClosedTime': None, 'Incide

2. Query `SecurityIncident` with KQL filters

In [21]:
incident_id = 2002
request_body = RunHuntingQueryPostRequestBody(
	query = f"SecurityIncident | where ProviderIncidentId == {incident_id} and TimeGenerated >= ago(7d)",
)
hunt_result = await graph_client.security.microsoft_graph_security_run_hunting_query.post(request_body)

INFO:httpx:HTTP Request: POST https://graph.microsoft.com/v1.0//security/microsoft.graph.security.runHuntingQuery "HTTP/1.1 200 OK"


In [22]:
print(vars(hunt_result).keys())

dict_keys(['backing_store', 'additional_data', 'odata_type', 'results', 'schema'])


In [23]:
print(hunt_result.results)

[HuntingRowResult(additional_data={'TenantId': UUID('f119dae4-df67-44a1-b5c7-caa589bcc8ce'), 'TimeGenerated': datetime.datetime(2026, 8, 11, 6, 56, 51, 20856, tzinfo=datetime.timezone.utc), 'IncidentName': UUID('1f5cf3b6-cfa5-4ae0-bac9-c2447236449d'), 'Title': 'Anomalous Windows Authentication Failure Activity', 'Description': 'An abnormal volume of Windows 4625 failed logon events is identified with look back over the past 24 hours.\n\nUsing _series_decompose_anomalies()_ with default values:\n- _threshold_ = 1.5 (for detecting mild or stronger anomalies)\n- _seasonality_ = -1 (autodetect seasonality using _series_periods_detect_)\n- _trend_ = _linefit_ (extract trend component using linear regression)', 'Severity': 'Medium', 'Status': 'New', 'Classification': '', 'ClassificationComment': '', 'ClassificationReason': '', 'Owner': '{"objectId":null,"email":null,"assignedTo":null,"userPrincipalName":null}', 'ProviderName': 'Microsoft XDR', 'ProviderIncidentId': '2002', 'FirstActivityTime

3. Cross workspace query:

In [24]:
request_body = RunHuntingQueryPostRequestBody(
	query = "workspace('alpha-soc').Syslog | where SyslogMessage contains 'failed password'",
    timespan = "P7D",
)
hunt_result = await graph_client.security.microsoft_graph_security_run_hunting_query.post(request_body)

INFO:httpx:HTTP Request: POST https://graph.microsoft.com/v1.0//security/microsoft.graph.security.runHuntingQuery "HTTP/1.1 200 OK"


In [25]:
print(vars(hunt_result).keys())

dict_keys(['backing_store', 'additional_data', 'odata_type', 'results', 'schema'])


In [26]:
print(hunt_result.results)

[HuntingRowResult(additional_data={'TenantId': UUID('5bc2e6fa-f7ac-4f50-8c88-da5a3ddd6e56'), 'SourceSystem': 'LogsIngestionAPI', 'TimeGenerated': datetime.datetime(2026, 8, 11, 4, 35, 5, 497311, tzinfo=datetime.timezone.utc), 'Computer': 'vm-alpha-ubuntu', 'EventTime': datetime.datetime(2026, 8, 11, 4, 34, 42, 746024, tzinfo=datetime.timezone.utc), 'Facility': 'auth', 'HostName': 'vm-alpha-ubuntu', 'SeverityLevel': 'info', 'SyslogMessage': 'Failed password for invalid user admin from 85.137.53.71 port 64928 ssh2', 'ProcessID@odata.type': '#Int64', 'ProcessID': 4245, 'HostIP': '10.1.0.4', 'ProcessName': 'sshd', 'MG': '', 'CollectorHostName': '', 'Type': 'Syslog', '_ResourceId': ''}, odata_type=None), HuntingRowResult(additional_data={'TenantId': UUID('5bc2e6fa-f7ac-4f50-8c88-da5a3ddd6e56'), 'SourceSystem': 'LogsIngestionAPI', 'TimeGenerated': datetime.datetime(2026, 8, 11, 4, 35, 5, 497311, tzinfo=datetime.timezone.utc), 'Computer': 'vm-alpha-ubuntu', 'EventTime': datetime.datetime(2026

### 3.4. Create comment for incident
The create comment API doesn't exist for python, a workaround is mentioned in [issue 624](https://github.com/microsoftgraph/msgraph-sdk-python/issues/624) that uses the `AlertComment` object to post as incident comment.

In [27]:
incident_id = 2002
request_body = AlertComment(
	odata_type = None,
	comment = f"Test set resolved {datetime.now().astimezone()}",
)
await graph_client.security.incidents.with_url(f"https://graph.microsoft.com/v1.0/security/incidents/{incident_id}/comments").post(request_body)

INFO:httpx:HTTP Request: POST https://graph.microsoft.com/v1.0/security/incidents/2002/comments "HTTP/1.1 200 OK"


Incident(additional_data={'@odata.context': "https://graph.microsoft.com/v1.0/$metadata#security/incidents('2002')/comments", 'value': [{'comment': 'Test set resolved 2026-08-11 13:59:14.577198+00:00', 'createdByDisplayName': 'API-App:autosoc@MngEnvMCAP398230.onmicrosoft.com', 'createdDateTime': datetime.datetime(2026, 8, 11, 13, 59, 14, 879544, tzinfo=datetime.timezone.utc)}, {'comment': 'Test Microsoft Graph SDK for Python', 'createdByDisplayName': 'API-App:autosoc@MngEnvMCAP398230.onmicrosoft.com', 'createdDateTime': datetime.datetime(2026, 8, 11, 13, 58, 22, 359137, tzinfo=datetime.timezone.utc)}, {'comment': 'Test Microsoft Graph SDK for Python', 'createdByDisplayName': 'API-App:autosoc@MngEnvMCAP398230.onmicrosoft.com', 'createdDateTime': datetime.datetime(2026, 8, 11, 13, 51, 14, 911476, tzinfo=datetime.timezone.utc)}, {'comment': 'Test Microsoft Graph SDK for Python', 'createdByDisplayName': 'API-App:autosoc2@MngEnvMCAP398230.onmicrosoft.com', 'createdDateTime': datetime.dateti

### 3.5. Update incident

In [28]:
incident_id = 2002
request_body = Incident(
    assigned_to = "Security Operations",
	classification = AlertClassification.InformationalExpectedActivity,
	determination = AlertDetermination.SecurityTesting,
    status = IncidentStatus.Resolved,
	custom_tags = [
        "Agent Triaged",
	],
    resolving_comment = f"Test set resolved {datetime.now().astimezone()}"
)
await graph_client.security.incidents.by_incident_id(incident_id).patch(request_body)

INFO:httpx:HTTP Request: PATCH https://graph.microsoft.com/v1.0//security/incidents/2002 "HTTP/1.1 200 OK"


Incident(additional_data={'@odata.context': 'https://graph.microsoft.com/v1.0/$metadata#security/incidents/$entity'}, id='2002', odata_type=None, alerts=None, assigned_to='Security Operations', classification=<AlertClassification.InformationalExpectedActivity: 'informationalExpectedActivity'>, comments=[AlertComment(additional_data={}, comment='Test set resolved 2026-08-11 13:59:14.577198+00:00', created_by_display_name='API-App:autosoc@MngEnvMCAP398230.onmicrosoft.com', created_date_time=datetime.datetime(2026, 8, 11, 13, 59, 14, 879544, tzinfo=datetime.timezone.utc), odata_type=None), AlertComment(additional_data={}, comment='Test Microsoft Graph SDK for Python', created_by_display_name='API-App:autosoc@MngEnvMCAP398230.onmicrosoft.com', created_date_time=datetime.datetime(2026, 8, 11, 13, 58, 22, 359137, tzinfo=datetime.timezone.utc), odata_type=None), AlertComment(additional_data={}, comment='Test Microsoft Graph SDK for Python', created_by_display_name='API-App:autosoc@MngEnvMCAP3

## Appendix on ISO 8601 time format

`(datetime.now().astimezone() - timedelta(hours=24)).isoformat()` generates an ISO 8601 **timestamp** in `YYYY-MM-DDTHH:mm:ss.ffffffZ` format

`P7D` (7 days), `PT24H` (24 hours) and `PT30M` (30 miniutes) are  ISO 8601 **time duration** format

In [29]:
print((datetime.now().astimezone() - timedelta(hours=24)).isoformat())

2026-08-11T14:01:18.900610+00:00


## Appendix 2: Threat Intelligence API test
Currently in appendix because TI API access is not availalbe yet

In [30]:
await graph_client.security.threat_intelligence.articles.get()

INFO:httpx:HTTP Request: GET https://graph.microsoft.com/v1.0//security/threatIntelligence/articles "HTTP/1.1 403 Forbidden"


ODataError: 
        APIError
        Code: 403
        message: None
        error: MainError(additional_data={}, code='notAllowed', details=None, inner_error=InnerError(additional_data={}, client_request_id='df4499be-5895-4eca-ac05-d73cb4a4b0ad', date=datetime.datetime(2026, 8, 12, 14, 1, 20), odata_type=None, request_id='51604448-5e27-4650-9a24-b5e4b96f041b'), message='Your account does not have access to this report or data. Please contact your organization administrator to ensure you have a valid license.', target=None)
        

In [ ]:
await graph_client.security.threat_intelligence.vulnerabilities.by_vulnerability_id('CVE-2026-32202').components.get()